<a href="https://colab.research.google.com/github/daniel-apolinario/puc-gestao-dados-saude/blob/main/estudo_caso_ultima_versao_melhorada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**🛠️ Passo 1: Configuração do Ambiente e Instalação de Dependências**

Para que o sistema de processamento de documentos e anonimização funcione corretamente, precisamos instalar bibliotecas específicas que não vêm pré-instaladas no Colab.

Nesta etapa, estamos configurando:

**LangChain & Groq:** Para a lógica da IA e interface com o modelo de linguagem.

**PyMuPDF:** Para leitura de arquivos PDF.

**Microsoft Presidio:** Ferramenta de ponta para detecção e anonimização de PII (Personally Identifiable Information).

**Spacy**: Baixamos o modelo en_core_web_lg para garantir maior precisão na identificação de entidades de texto.

**Nota:** É comum aparecerem avisos de conflito de dependências (em vermelho) durante a instalação do pip. Você pode ignorá-los, pois as funcionalidades principais necessárias para este projeto não serão afetadas.

In [ ]:
!pip install -q -U langchain-groq langchain_core pandas pymupdf presidio-analyzer presidio-anonymizer
# Importante: O Presidio precisa carregar um modelo de NLP para funcionar
!python -m spacy download en_core_web_lg -q

print("✅ Bibliotecas instaladas. Se houver avisos em vermelho sobre 'pip dependency resolver', "
      "pode ignorar, pois as bibliotecas que usaremos (LangChain/Presidio) estarão funcionais.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Bibliotecas instaladas. Se houver avisos em vermelho sobre 'pip dependency resolver', pode ignorar, pois as bibliotecas que usaremos (LangChain/Presidio) estarão funcionais.


**🔑 Passo 2: Configuração de Credenciais e Chaves de API**

Para utilizar os modelos de linguagem de alto desempenho (LLMs) via Groq, precisamos autenticar nossa sessão.

Nesta célula:

Importamos as ferramentas necessárias para manipular variáveis de ambiente.

Configuramos a chave de acesso da API.

**Segurança dos Dados:** No Google Colab, recomenda-se utilizar o recurso de "Secrets" (ícone da chave na barra lateral esquerda) para armazenar chaves sensíveis. Assim, você pode chamá-las usando userdata.get('NOME_DA_CHAVE') em vez de expor o texto da chave diretamente no script, garantindo que sua conta não seja utilizada indevidamente por terceiros ao compartilhar o notebook.

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('grok_api_key')

**📄 Passo 3: Criação do "Documento de Teste" (Simulação de Laudo)**

Para testarmos nossas ferramentas de IA e privacidade, precisamos de um documento de exemplo. Nesta etapa, vamos automatizar a criação de um arquivo PDF que simula um laudo radiológico contendo **Informações de Identificação Pessoal (PII)**.

O que este trecho faz:

**Instala o ReportLab:** Biblioteca necessária para gerar arquivos PDF programaticamente.

**Define a Função criar_pdf:** Uma função que converte listas de texto em páginas de documento formatadas.

**Gera o Laudo Exemplo:** Criamos o arquivo laudo_ocr_sujo.pdf contendo dados sensíveis propositais (Nome, CPF e Achados Clínicos) para validarmos o processo de anonimização nas células seguintes.

In [ ]:
!pip install reportlab -q

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import os

def criar_pdf(nome_arquivo, conteudo):
    c = canvas.Canvas(nome_arquivo, pagesize=letter)
    textobject = c.beginText(50, 750)
    textobject.setFont("Helvetica", 10)

    for linha in conteudo:
        textobject.textLine(linha)

    c.drawText(textobject)
    c.save()
    print(f"Arquivo {nome_arquivo} gerado com sucesso!")

# Simulando um texto de laudo que veio de um PDF
texto_sujo = [
"HOSPITAL DELTA - LAUDO RADIOLÓGICO",
"PACIENTE: João da Silva Santos  CPF: 123.456.789-00",
"DATA: 17/03/2026",
"ACHADOS: Observa-se massa expansiva em lobo superior direito de 4cm.",
"Sugestivo de neoplasia maligna. Paciente apresenta dispneia grave."
]

# Gerar os arquivos
criar_pdf("laudo_sujo.pdf", texto_sujo)

Arquivo laudo_sujo.pdf gerado com sucesso!


**📂 Passo 4: Extração de Texto do PDF **

Nesta etapa, realizamos a leitura técnica do arquivo digital para extrair seu conteúdo textual bruto. Em arquiteturas de dados, chamamos essa fase inicial de "Camada Bronze", onde os dados são capturados exatamente como estão na fonte.

O que este trecho faz:

**Define a Função de Extração:** Criamos uma lógica que abre o PDF e "varre" todas as páginas em busca de texto.

**Validação de Arquivo:** O script verifica se o arquivo está presente no ambiente do Colab antes de tentar a leitura.


In [ ]:
import fitz  # PyMuPDF

def extrair_texto_pdf_real(caminho_arquivo):
    try:
        # 1. Abre o documento PDF
        doc = fitz.open(caminho_arquivo)
        texto_completo = ""

        # 2. Itera por todas as páginas (laudos podem ter mais de uma página)
        for pagina in doc:
            texto_completo += pagina.get_text()

        doc.close()
        return texto_completo
    except Exception as e:
        return f"Erro ao ler o PDF: {e}"

# --- TESTE NA LIVE ---
# 1. Faça o upload de um arquivo 'laudo.pdf' no menu lateral do Colab
# 2. Altere o nome abaixo para o nome do seu arquivo
caminho = "laudo_sujo.pdf"

# Verificação se o arquivo existe antes de tentar ler (boa prática de live)
import os
if os.path.exists(caminho):
    laudo_bruto = extrair_texto_pdf_real(caminho)
    print("--- Conteúdo do PDF Real (Bronze) ---")
    print(laudo_bruto[:1000]) # Mostra os primeiros 1000 caracteres
else:
    print(f"⚠️ O arquivo '{caminho}' não foi encontrado. Por favor, faça o upload no menu lateral.")
    # Fallback para ter um texto de exemplo caso ocorra algum problema com o PDF
    laudo_bruto = "HOSPITAL DELTA\nPaciente: Carlos Exemplo\nAchado: Nódulo em pulmão."

--- Conteúdo do PDF Real (Bronze) ---
HOSPITAL DELTA - LAUDO RADIOLÓGICO
PACIENTE: João da Silva Santos  CPF: 123.456.789-00
DATA: 17/03/2026
ACHADOS: Observa-se massa expansiva em lobo superior direito de 4cm.
Sugestivo de neoplasia maligna. Paciente apresenta dispneia grave.



**🛡️ Passo 5: Proteção de Dados e Anonimização**

Em cenários de saúde, a segurança da informação é mandatória. Nesta célula, utilizamos o Microsoft Presidio para identificar e mascarar informações sensíveis (PII - Personally Identifiable Information).

O que este trecho faz:

**Inicializa os Motores de IA:** Configuramos o analisador (que detecta os dados) e o anonimizador (que os mascara).

**Análise de Entidades**: O sistema varre o laudo bruto em busca de nomes próprios e números de identificação.

**Geração de dados anonimizados:** O texto original é transformado. Agora temos um Dado Anonimizado, pronto para ser processado por modelos de inteligência artificial em nuvem com riscos de vazamento de privacidade drasticamente reduzidos.

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

results = analyzer.analyze(text=laudo_bruto, entities=["PERSON", "ID"], language='en') # Adaptar para PT conforme necessidade
laudo_anonimo = anonymizer.anonymize(text=laudo_bruto, analyzer_results=results)

print("--- Texto Anonimizado (Silver) ---")
print(laudo_anonimo.text)

--- Texto Anonimizado (Silver) ---
HOSPITAL DELTA - LAUDO RADIOLÓGICO
PACIENTE: <PERSON>  CPF: 123.456.789-00
DATA: 17/03/2026
ACHADOS: Observa-se massa expansiva em lobo superior direito de 4cm.
Sugestivo de neoplasia maligna. <PERSON> apresenta dispneia grave.



**🇧🇷 Passo 6: Customização para o Contexto Brasileiro (Regex & CPF)**

Ferramentas globais nem sempre reconhecem padrões específicos de cada país, como o formato do nosso CPF. Nesta etapa, personalizamos a inteligência do sistema para garantir que nenhum dado sensível brasileiro passe despercebido.

O que este trecho faz:

**Cria um Padrão (Regex)**: Ensinamos ao script a estrutura matemática de um CPF (pontos e traços).

**Injeção de Lógica:** Adicionamos esse "conhecimento" ao motor de análise do Presidio.

**Refinamento da Anonimização:** O sistema agora é capaz de identificar e mascarar tanto nomes próprios quanto CPFs de forma cirúrgica, elevando o nível de conformidade do dado para a Camada Silver.

In [ ]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine

# 1. Criar um reconhecedor customizado para o CPF Brasileiro
cpf_pattern = Pattern(name="cpf_pattern", regex=r"\d{3}\.\d{3}\.\d{3}-\d{2}", score=0.5)
cpf_recognizer = PatternRecognizer(supported_entity="CPF", patterns=[cpf_pattern])

# 2. Inicializar o motor e adicionar o novo reconhecedor
analyzer = AnalyzerEngine()
analyzer.registry.add_recognizer(cpf_recognizer)
anonymizer = AnonymizerEngine()

# 3. Analisar (pedindo especificamente para buscar PERSON e o nosso novo CPF)
# Nota: Ignoramos os warnings desativando o log se desejar, mas vamos focar no resultado
results = analyzer.analyze(text=laudo_bruto, entities=["PERSON", "CPF"], language='en')

# 4. Anonimizar
laudo_anonimo = anonymizer.anonymize(text=laudo_bruto, analyzer_results=results)

print("--- Texto Anonimizado com Sucesso (Silver) ---")
print(laudo_anonimo.text)

--- Texto Anonimizado com Sucesso (Silver) ---
HOSPITAL DELTA - LAUDO RADIOLÓGICO
PACIENTE: <PERSON>  CPF: <CPF>
DATA: 17/03/2026
ACHADOS: Observa-se massa expansiva em lobo superior direito de 4cm.
Sugestivo de neoplasia maligna. <PERSON> apresenta dispneia grave.



**🦙 Passo 7: Extração de Insights com Llama 3 e LangChain**

Chegamos à etapa final do pipeline de dados. Agora que o dado está seguro e anonimizado, utilizamos um Modelo de Linguagem de Larga Escala (LLM) de última geração para extrair inteligência clínica estruturada.

O que este trecho faz:

**Configura o Llama 3.3**: Utilizamos a infraestrutura da Groq para processamento em milissegundos.

**Define a Cadeia de Raciocínio (Chain):** Criamos um fluxo que recebe o laudo anonimizado e o processa sob a ótica de um especialista em dados de saúde.

**Estruturação JSON:** A IA converte o texto livre em um formato de dados (JSON) contendo Diagnóstico, Gravidade e códigos de terminologia médica (CID-10 e SNOMED-CT).

**Transformação em Camada Gold:** O resultado final não é mais apenas texto, mas sim conhecimento estruturado, pronto para alimentar sistemas de suporte à decisão clínica ou painéis de gestão hospitalar.

**🧠 O "Raciocínio" por trás da Classificação de Risco**
É importante notar que a classificação de Gravidade e a sugestão de CID-10 não são campos extraídos diretamente do texto (como um "copia e cola"), mas sim uma inferência clínica realizada pelo modelo Llama 3.3-70b.

**Como a IA chega a essa conclusão?**

**Processamento Semântico:** O modelo identifica "neoplasia maligna" e "dispneia grave" como indicadores de criticidade.

**Alinhamento de Especialista:** Graças ao System Prompt que definimos, a IA filtra as informações sob a ótica de um profissional de dados em saúde, priorizando a segurança do paciente na estruturação do JSON.

**Interoperabilidade:** A IA atua como uma ponte, traduzindo a linguagem natural do médico para padrões globais como SNOMED-CT, permitindo que o dado "fale a mesma língua" que outros sistemas hospitalares.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Instanciando o Llama 3 (O modelo open-source mais potente do mundo hoje)
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

# 2. Prompt Estruturado para o Hospital Delta
template = """
Você é um Arquiteto de Dados em Saúde.
As tags <PERSON> e <CPF> representam dados privados anonimizados.

TAREFA:
Analise o laudo e extraia um JSON com:
- Diagnostico_Principal
- Gravidade (Baixa, Média, Alta)
- CID_10_Provavel
- Termo_SNOMED_CT
- Internacao_Imediata (Sim/Não)

Laudo: {laudo}

RETORNE APENAS O JSON:
"""

prompt = PromptTemplate(input_variables=["laudo"], template=template)

# 3. Chain (Sintaxe LCEL - Mantendo o requisito da aula)
chain = prompt | llm | JsonOutputParser()

# 4. Execução
try:
    resultado_json = chain.invoke({"laudo": laudo_anonimo.text})
    print("--- 🦙 Insights Extraídos com Llama 3 ---")
    import json
    print(json.dumps(resultado_json, indent=4, ensure_ascii=False))
except Exception as e:
    print(f"❌ Erro: {e}")

--- 🦙 Insights Extraídos com Llama 3 ---
{
    "Diagnostico_Principal": "Neoplasia Maligna",
    "Gravidade": "Alta",
    "CID_10_Provavel": "C34.1",
    "Termo_SNOMED_CT": "Neoplasia maligna de pulmão",
    "Internacao_Imediata": "Sim"
}


**🏥 Passo 8: Interoperabilidade com Padrão HL7 FHIR**

Para que os dados gerados pela IA sejam úteis na prática clínica, eles precisam "falar a mesma língua" que os sistemas hospitalares. Nesta etapa final, convertemos nossos insights para o padrão HL7 FHIR, o padrão ouro global para troca de informações de saúde.

O que este trecho faz:

**Mapeamento para Recurso Condition:** Organizamos as informações de diagnóstico e gravidade seguindo a especificação técnica do FHIR.

**Vinculação de Vocabulários Controlados:** Atribuímos as URIs oficiais para os códigos CID-10 e SNOMED-CT, garantindo que o dado seja semanticamente correto.

**Geração do Objeto de Interoperabilidade**: O resultado final é um recurso pronto para ser integrado a Prontuários Eletrônicos do Paciente (PEP) ou enviado para Redes de Atenção à Saúde.

In [ ]:
# Transformação para HL7 FHIR
if 'resultado_json' in locals():
    # Mapeamento simples para o padrão FHIR Condition
    recurso_fhir = {
        "resourceType": "Condition",
        "severity": {"text": resultado_json.get("Gravidade")},
        "code": {
            "coding": [
                {"system": "http://hl7.org/fhir/sid/icd-10", "code": resultado_json.get("CID_10_Provavel")},
                {"system": "http://snomed.info/sct", "display": resultado_json.get("Termo_SNOMED_CT")}
            ]
        },
        "note": [{"text": f"Internação: {resultado_json.get('Internacao_Imediata')}"}]
    }
    print("--- 🏁 Recurso FHIR Gerado com Sucesso ---")
    print(json.dumps(recurso_fhir, indent=4, ensure_ascii=False))

--- 🏁 Recurso FHIR Gerado com Sucesso ---
{
    "resourceType": "Condition",
    "severity": {
        "text": "Alta"
    },
    "code": {
        "coding": [
            {
                "system": "http://hl7.org/fhir/sid/icd-10",
                "code": null
            },
            {
                "system": "http://snomed.info/sct",
                "display": "Neoplasia maligna de pulmão"
            }
        ]
    },
    "note": [
        {
            "text": "Internação: Sim"
        }
    ]
}


**🚀 Passo 9: Integração Real - Enviando para um Servidor FHIR (Sandbox)**

Para encerrar nosso fluxo, vamos realizar uma operação de Interoperabilidade Ativa. Não vamos apenas gerar o dado, mas sim "conversar" com um servidor de saúde real (Sandbox público) para persistir nossa informação.

O que este trecho faz:

**Estabelece Conexão:** Conecta o Google Colab ao servidor público HAPI FHIR (versão R4), utilizado mundialmente por desenvolvedores de sistemas de saúde.

**Realiza o Envio (POST):** Transmite nosso recurso anonimizado e estruturado via protocolo seguro.

**Valida a Persistência:** O servidor processa o JSON e nos devolve um ID de transação.

**Link de Visualização:** O código gera um link direto onde você pode ver o seu laudo "vivo" no servidor, simulando o que ocorreria em uma integração entre um laboratório e o Prontuário Eletrônico do Paciente (PEP).

In [ ]:
import requests
import json

def enviar_para_sandbox_fhir(recurso_fhir):
    # Endpoint público do HAPI FHIR (Versão R4)
    url_sandbox = "http://hapi.fhir.org/baseR4/Condition"

    headers = {
        "Content-Type": "application/fhir+json",
        "Accept": "application/fhir+json"
    }

    print(f"📡 Enviando recurso para o Servidor Público HAPI FHIR...")

    try:
        # Envio Real via POST
        response = requests.post(url_sandbox, json=recurso_fhir, headers=headers, timeout=15)

        if response.status_code in [200, 201]:
            dados_retorno = response.json()
            server_id = dados_retorno.get("id")
            print(f"✅ SUCESSO! O recurso foi aceito pelo servidor.")
            print(f"🆔 ID Gerado no Servidor: {server_id}")
            print(f"🔗 Link para visualizar: http://hapi.fhir.org/baseR4/Condition/{server_id}")
        else:
            print(f"❌ Erro no Servidor: {response.status_code}")
            print(response.text)

    except Exception as e:
        print(f"❌ Falha na conexão: {e}")

# --- EXECUÇÃO ---
if 'recurso_fhir' in locals():
    enviar_para_sandbox_fhir(recurso_fhir)

📡 Enviando recurso para o Servidor Público HAPI FHIR...
❌ Erro no Servidor: 412
{
  "resourceType": "OperationOutcome",
  "text": {
    "status": "generated",
    "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><h1>Operation Outcome</h1><table border=\"0\"><tr><td style=\"font-weight: bold;\">ERROR</td><td>[]</td><td>HAPI-2840: Can not create resource duplicating existing resource: Condition/131415214</td></tr></table></div>"
  },
  "issue": [ {
    "severity": "error",
    "code": "processing",
    "diagnostics": "HAPI-2840: Can not create resource duplicating existing resource: Condition/131415214"
  } ]
}


**📊 Passo 10: Dashboard de Priorização e Suporte à Decisão**

De nada serve o dado estruturado se ele não chegar ao tomador de decisão de forma clara. Nesta etapa final, criamos um Painel de Risco Dinâmico que consome o recurso FHIR e aplica regras de negócio hospitalar para triagem automática.

O que este trecho faz:

**Extração de Metadados:** Recupera de forma segura os códigos CID e SNOMED-CT de dentro da estrutura FHIR.

**Motor de Regras:** Avalia a severidade do caso e define visualmente o nível de alerta (Verde para estável, Vermelho para emergência).

**Renderização de UI:** Gera uma interface visual elegante (HTML/CSS) diretamente no notebook, simulando o que um gestor hospitalar veria em sua tela de monitoramento.

**Conclusão do Ciclo**: Demonstra a jornada completa do dado: desde a recepção de um documento não estruturado até a geração de um insight crítico para salvar vidas.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

def gerar_painel_priorizacao(recurso_fhir):
    # 1. Extração Segura de metadados (Usando .get para evitar KeyError)
    cid_coding = recurso_fhir.get("code", {}).get("coding", [{}])[0]
    snomed_coding = recurso_fhir.get("code", {}).get("coding", [{}])[1]

    diagnostico = snomed_coding.get("display", "Diagnóstico não identificado")
    cid = cid_coding.get("code", "R69") # R69 é o CID genérico para 'desconhecido'

    # Extração de gravidade e notas
    gravidade = recurso_fhir.get("severity", {}).get("text", "Média")

    notas = recurso_fhir.get("note", [{}])[0].get("text", "Internação: Não")
    internacao = "Não"
    if "Sim" in notas:
        internacao = "Sim"

    timestamp = recurso_fhir.get("recordedDate", "Horário não registrado")

    # 2. Lógica de Negócio Hospitalar (Visualização)
    cor_alerta = "green"
    status = "Estável"

    # Lógica de priorização baseada nos dados extraídos
    if gravidade.upper() in ["ALTA", "HIGH"] or internacao == "Sim":
        cor_alerta = "red"
        status = "🚨 EMERGÊNCIA IMEDIATA"
    elif gravidade.upper() in ["MÉDIA", "MEDIA", "MEDIUM"]:
        cor_alerta = "orange"
        status = "⚠️ ATENÇÃO PRIORITÁRIA"

    # 3. HTML do Dashboard para a Live
    html_painel = f"""
    <div style="padding: 20px; border: 3px solid {cor_alerta}; border-radius: 12px; background-color: #ffffff; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 5px 5px 15px rgba(0,0,0,0.1);">
        <h2 style="color: {cor_alerta}; margin-top: 0; border-bottom: 2px solid {cor_alerta}; padding-bottom: 10px;">Painel de Risco - Hospital Delta</h2>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px;">
            <tr style="border-bottom: 1px solid #eee;"><td style="font-weight: bold; padding: 8px;">STATUS DO PACIENTE:</td><td style="color: {cor_alerta}; font-weight: bold; padding: 8px;">{status}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="font-weight: bold; padding: 8px;">ACHADO CLÍNICO:</td><td style="padding: 8px;">{diagnostico}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="font-weight: bold; padding: 8px;">CÓDIGO CID-10:</td><td style="padding: 8px;"><code>{cid}</code></td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="font-weight: bold; padding: 8px;">GRAVIDADE (LLM):</td><td style="padding: 8px;">{gravidade.upper()}</td></tr>
            <tr><td style="font-weight: bold; padding: 8px;">DATA/HORA:</td><td style="padding: 8px; font-size: 0.9em; color: #666;">{timestamp}</td></tr>
        </table>
        <div style="margin-top: 15px; background: #f0f0f0; padding: 10px; border-radius: 5px; font-size: 0.85em;">
            <strong>📌 Decisão de Arquitetura:</strong> Dado extraído via Agente de IA e estruturado em padrão HL7 FHIR para interoperabilidade.
        </div>
    </div>
    """
    display(HTML(html_painel))

# --- EXECUÇÃO ---
print("Simulando o consumo do Repositório FHIR...")
if 'recurso_fhir' in locals():
    gerar_painel_priorizacao(recurso_fhir)
else:
    print("⚠️ Erro: Recurso FHIR não encontrado.")

Simulando o consumo do Repositório FHIR...


STATUS DO PACIENTE:,🚨 EMERGÊNCIA IMEDIATA
ACHADO CLÍNICO:,Neoplasia maligna de pulmão
CÓDIGO CID-10:,None
GRAVIDADE (LLM):,ALTA
DATA/HORA:,Horário não registrado
